<a href="https://colab.research.google.com/github/Snowpuppies2020/Finance/blob/main/7_7%E5%8D%8A%E5%B0%8E%E4%BD%93%E9%96%A2%E9%80%A3%E3%81%8B%E3%82%89%E3%81%AE%E8%B3%87%E9%87%91%E3%83%AD%E3%83%BC%E3%83%86%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
半導体/メモリ -> ソフトウェア ローテーション戦略 バックテスト
=================================================================

背景:
2026年は年初来、ハイパースケーラーのAI Capex拡大とメモリ需給ひっ迫を
背景に半導体・メモリ株(SOX指数)がソフトウェア株(IGV等)に対し大幅アウト
パフォームしてきたが、2026年7月頭にハイパースケーラーの「Capex過剰投資
懸念」を引き金に、初めて明確な資金シフト(半導体売り/ソフトウェア買い)
が観測された。本スクリプトはこの現象を「相対モメンタムのトレンド転換」
としてシグナル化し、過去の類似局面を含めてローテーション戦略を検証する。

使い方:
    pip install yfinance pandas numpy matplotlib --break-system-packages
    python semi_to_software_rotation_backtest.py

Bloomberg Terminal を使う場合は `fetch_prices()` を BQL/BDH 呼び出しに
置き換えてください（コメント参照）。
"""

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# 1. ユニバース定義
# ------------------------------------------------------------------
SEMI_MEMORY_BASKET = [
    "MU", "WDC", "SNDK",          # メモリ
    "TSM", "AMAT", "KLAC", "ASML",  # 半導体製造装置/ファウンドリ
    # 日本株を混ぜる場合は "8035.T", "6857.T", "6146.T", "285A.T" を追加
]

SOFTWARE_BASKET = [
    "MSFT", "ORCL", "CRM", "ADBE", "NOW",
    "PANW", "CRWD", "PLTR", "APP",
]

BENCH_SEMI_ETF = "SOXX"   # 比較用（実データがなければ自前バスケットで代替可）
BENCH_SW_ETF = "IGV"

START = "2018-01-01"
END = None  # None = today


# ------------------------------------------------------------------
# 2. データ取得（Bloomberg利用時はここを差し替え）
# ------------------------------------------------------------------
def fetch_prices(tickers, start=START, end=END):
    """
    yfinance経由で調整後終値を取得。
    Bloombergを使う場合の置き換え例:
        from xbbg import blp
        df = blp.bdh(tickers, 'PX_LAST', start, end)
    """
    import yfinance as yf
    data = yf.download(tickers, start=start, end=end, auto_adjust=True)["Close"]
    if isinstance(data, pd.Series):
        data = data.to_frame()
    return data.dropna(how="all")


def basket_index(prices: pd.DataFrame) -> pd.Series:
    """等ウェイトで正規化したバスケット指数(基準=100)を作る"""
    norm = prices / prices.iloc[0] * 100
    return norm.mean(axis=1)


# ------------------------------------------------------------------
# 3. シグナル生成: 相対力(Relative Strength)のMAクロス
# ------------------------------------------------------------------
def build_signal(semi_idx: pd.Series, sw_idx: pd.Series,
                  fast: int = 20, slow: int = 100) -> pd.Series:
    """
    ratio = 半導体指数 / ソフトウェア指数
    fast MA が slow MA を下抜け -> 半導体劣後シグナル(=ソフトウェアへローテーション, +1)
    fast MA が slow MA を上抜け -> 半導体優位シグナル(=半導体へローテーション, -1)
    """
    ratio = semi_idx / sw_idx
    fast_ma = ratio.rolling(fast).mean()
    slow_ma = ratio.rolling(slow).mean()

    signal = pd.Series(np.where(fast_ma > slow_ma, -1, 1), index=ratio.index)
    # ゴールデン/デッドクロス直後だけシグナルを立てたい場合はdiffで検出も可
    return signal.reindex(ratio.index).ffill()


# ------------------------------------------------------------------
# 4. バックテスト本体
# ------------------------------------------------------------------
def backtest(semi_idx, sw_idx, signal, cost_bps=10):
    """
    signal == +1 : ソフトウェアをロング（半導体はキャッシュ or ショート）
    signal == -1 : 半導体をロング（ソフトウェアはキャッシュ or ショート）
    ロング・ショート型にしたい場合は long_short=True に拡張してください。
    """
    semi_ret = semi_idx.pct_change()
    sw_ret = sw_idx.pct_change()

    # シグナルは翌日始値でエントリーと仮定 → 1日ラグ
    sig = signal.shift(1)

    strat_ret = np.where(sig == 1, sw_ret, semi_ret)
    strat_ret = pd.Series(strat_ret, index=semi_ret.index)

    # 売買コスト: シグナル変化日にコストを差し引く
    turnover = sig.diff().abs().fillna(0) / 2
    cost = turnover * (cost_bps / 10000)
    strat_ret = strat_ret - cost

    strat_ret = strat_ret.dropna()
    return strat_ret


def perf_stats(returns: pd.Series, freq=252):
    cum = (1 + returns).cumprod()
    total_return = cum.iloc[-1] - 1
    ann_return = (1 + total_return) ** (freq / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(freq)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    dd = (cum / cum.cummax() - 1).min()
    return {
        "累積リターン": f"{total_return:.1%}",
        "年率リターン": f"{ann_return:.1%}",
        "年率ボラティリティ": f"{ann_vol:.1%}",
        "シャープレシオ": f"{sharpe:.2f}",
        "最大ドローダウン": f"{dd:.1%}",
    }


# ------------------------------------------------------------------
# 5. 実行
# ------------------------------------------------------------------
if __name__ == "__main__":
    all_tickers = SEMI_MEMORY_BASKET + SOFTWARE_BASKET
    prices = fetch_prices(all_tickers)

    semi_idx = basket_index(prices[SEMI_MEMORY_BASKET])
    sw_idx = basket_index(prices[SOFTWARE_BASKET])

    for fast, slow in [(10, 50), (20, 100), (50, 200)]:
        signal = build_signal(semi_idx, sw_idx, fast, slow)
        strat_ret = backtest(semi_idx, sw_idx, signal, cost_bps=10)

        print(f"\n=== ローテーション戦略 (fast={fast}, slow={slow}) ===")
        for k, v in perf_stats(strat_ret).items():
            print(f"  {k}: {v}")

    # ベンチマーク: 半導体バスケットのバイ&ホールド / ソフトウェアのバイ&ホールド
    print("\n=== ベンチマーク: 半導体バスケット買い持ち ===")
    for k, v in perf_stats(semi_idx.pct_change().dropna()).items():
        print(f"  {k}: {v}")

    print("\n=== ベンチマーク: ソフトウェアバスケット買い持ち ===")
    for k, v in perf_stats(sw_idx.pct_change().dropna()).items():
        print(f"  {k}: {v}")

    print("\n【注意】")
    print("・上記は価格モメンタムのみを使った代理シグナルです。")
    print("・実際の『Capex懸念』を反映させるには、決算ガイダンスでの")
    print("  設備投資計画の下方修正日をイベントフラグとして追加し、")
    print("  シグナルと重ね合わせることを推奨します。")
    print("・2026年はSOXXがIGVに対し年初来で大幅優位という長いトレンドの")
    print("  最中での反転初期局面のため、ダマシ(whipsaw)のリスクが高い点に注意。")

[*********************100%***********************]  16 of 16 completed



=== ローテーション戦略 (fast=10, slow=50) ===
  累積リターン: 1878.1%
  年率リターン: 42.2%
  年率ボラティリティ: 35.1%
  シャープレシオ: 1.20
  最大ドローダウン: -50.6%

=== ローテーション戦略 (fast=20, slow=100) ===
  累積リターン: 1874.0%
  年率リターン: 42.2%
  年率ボラティリティ: 35.6%
  シャープレシオ: 1.18
  最大ドローダウン: -50.7%

=== ローテーション戦略 (fast=50, slow=200) ===
  累積リターン: 1124.2%
  年率リターン: 34.4%
  年率ボラティリティ: 36.1%
  シャープレシオ: 0.95
  最大ドローダウン: -54.4%

=== ベンチマーク: 半導体バスケット買い持ち ===
  累積リターン: 1486.2%
  年率リターン: 38.6%
  年率ボラティリティ: 39.2%
  シャープレシオ: 0.98
  最大ドローダウン: -48.4%

=== ベンチマーク: ソフトウェアバスケット買い持ち ===
  累積リターン: 396.5%
  年率リターン: 20.8%
  年率ボラティリティ: 30.0%
  シャープレシオ: 0.69
  最大ドローダウン: -40.9%

【注意】
・上記は価格モメンタムのみを使った代理シグナルです。
・実際の『Capex懸念』を反映させるには、決算ガイダンスでの
  設備投資計画の下方修正日をイベントフラグとして追加し、
  シグナルと重ね合わせることを推奨します。
・2026年はSOXXがIGVに対し年初来で大幅優位という長いトレンドの
  最中での反転初期局面のため、ダマシ(whipsaw)のリスクが高い点に注意。


In [2]:
def signal_turnover_per_year(signal: pd.Series) -> pd.Series:
    """
    シグナルが切り替わった日をカウントし、年ごとの切り替え回数を集計する。
    年10回以上ならダマシ(whipsaw)が多く実運用に不向きと判断する目安。
    """
    changes = signal.diff().fillna(0) != 0
    changes = changes[signal.index]  # インデックス整合
    per_year = changes.groupby(changes.index.year).sum()
    return per_year


# --- __main__ 内、各 (fast, slow) の検証ループに追加 ---
for fast, slow in [(10, 50), (20, 100), (50, 200)]:
    signal = build_signal(semi_idx, sw_idx, fast, slow)
    strat_ret = backtest(semi_idx, sw_idx, signal, cost_bps=10)

    print(f"\n=== ローテーション戦略 (fast={fast}, slow={slow}) ===")
    for k, v in perf_stats(strat_ret).items():
        print(f"  {k}: {v}")

    turnover = signal_turnover_per_year(signal.shift(1).dropna())
    print("  --- 年間シグナル切り替え回数 ---")
    print(turnover.to_string())
    print(f"  平均切り替え回数/年: {turnover.mean():.1f}回")



=== ローテーション戦略 (fast=10, slow=50) ===
  累積リターン: 1878.1%
  年率リターン: 42.2%
  年率ボラティリティ: 35.1%
  シャープレシオ: 1.20
  最大ドローダウン: -50.6%
  --- 年間シグナル切り替え回数 ---
Date
2018    4
2019    7
2020    4
2021    4
2022    8
2023    8
2024    3
2025    5
2026    0
  平均切り替え回数/年: 4.8回

=== ローテーション戦略 (fast=20, slow=100) ===
  累積リターン: 1874.0%
  年率リターン: 42.2%
  年率ボラティリティ: 35.6%
  シャープレシオ: 1.18
  最大ドローダウン: -50.7%
  --- 年間シグナル切り替え回数 ---
Date
2018    0
2019    3
2020    4
2021    2
2022    6
2023    1
2024    2
2025    3
2026    0
  平均切り替え回数/年: 2.3回

=== ローテーション戦略 (fast=50, slow=200) ===
  累積リターン: 1124.2%
  年率リターン: 34.4%
  年率ボラティリティ: 36.1%
  シャープレシオ: 0.95
  最大ドローダウン: -54.4%
  --- 年間シグナル切り替え回数 ---
Date
2018    0
2019    1
2020    2
2021    1
2022    5
2023    1
2024    2
2025    1
2026    0
  平均切り替え回数/年: 1.4回
